# Module 2: Tokenization (BPE, SentencePiece)

## Learning Objectives
- Understand how tokenizers work and why they matter
- Implement a minimal Byte-Pair Encoding (BPE) tokenizer
- Compare tokenization strategies and their effect on prompt length
- Connect BPE concepts to Kyro's LuminaTokenizer

In [ ]:
import tokenizers
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
import matplotlib.pyplot as plt
import numpy as np
print('tokenizers library loaded')

## Step 1: What is Tokenization?

In [ ]:
text = 'Hello, world! This is a test of tokenization.'
char_tokens = list(text)
word_tokens = text.split()
print(f'Character-level: {len(char_tokens)} tokens')
print(f'Word-level: {len(word_tokens)} tokens')
print('Subword-level (BPE/SentencePiece):')
print('  - Compresses common words into single tokens')
print('  - Handles OOV words by breaking them into subwords')
print('  - Balances vocabulary size and token count')

## Step 2: Build a Minimal BPE Tokenizer

In [ ]:
class MinimalBPE:
    def __init__(self):
        self.vocab = {}
        self.build_vocab()

    def build_vocab(self):
        chars = 'abcdefghijklmnopqrstuvwxyz ,.!?'
        for i, c in enumerate(chars):
            self.vocab[c] = i
        bigrams = ['he', 'th', 'in', 'er', 'an', 're', 'on', 'at', 'en', 'nd']
        for i, bg in enumerate(bigrams, start=len(chars)):
            self.vocab[bg] = i

    def encode(self, text):
        text = text.lower()
        tokens = []
        chars = list(text)
        i = 0
        while i < len(chars):
            matched = False
            for length in [3, 2, 1]:
                if i + length <= len(chars):
                    substr = ''.join(chars[i:i+length])
                    if substr in self.vocab:
                        tokens.append(self.vocab[substr])
                        i += length
                        matched = True
                        break
            if not matched:
                c = chars[i]
                tokens.append(self.vocab.get(c, 0))
                i += 1
        return tokens

    def decode(self, token_ids):
        inv_vocab = {v: k for k, v in self.vocab.items()}
        return ''.join(inv_vocab.get(tid, '?') for tid in token_ids)

bpe = MinimalBPE()
text = 'hello world'
tokens = bpe.encode(text)
print(f'Text: {text}')
print(f'Token IDs: {tokens}')
print(f'Detokenized: {bpe.decode(tokens)}')
print(f'Token count: {len(tokens)}')

## Step 3: Compare Tokenizers with the `tokenizers` Library

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

bpe_tokenizer = Tokenizer(models.BPE())
bpe_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.BpeTrainer(vocab_size=100, min_frequency=1, special_tokens=['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]'])
bpe_tokenizer.train_from_iterator(['hello world this is a test', 'the quick brown fox', 'machine learning is fun'])

text = 'hello world machine learning'
encoding = bpe_tokenizer.encode(text)
print(f'Text: {text}')
print(f'Token IDs: {encoding.ids}')
print(f'Tokens: {encoding.tokens}')
print(f'Token count: {len(encoding.ids)}')
print(f'Decoded: {bpe_tokenizer.decode(encoding.ids)}')

## Exercises
1. Build BPE merges: Implement the pair-counting and merging logic from scratch.
2. Compare vocab sizes: Train with different vocab sizes and compare token counts.
3. Implement detokenizer: Create the reverse mapping from token IDs to text.
4. Kyro connection: Explore `src/api/tokenizer.rs` — how does LuminaTokenizer.encode() work?